# 95b — Review rejected nodal events and stack QC

Read-only review of the products written by notebook 95. This notebook never modifies the SQLite catalog. It creates:

- a campaign-wide QC dashboard,
- an accepted/rejected position-versus-time figure,
- stack and event review queues as CSV files, and
- waveform panels for the highest-priority stacks plus a small good-stack audit sample.

Review the failed and high-rejection stacks first. In each waveform panel, accepted events are blue, rejected events are orange-red, and the consensus reference is black. Waveforms are aligned using the shifts computed by notebook 95.

In [1]:
from pathlib import Path
import sqlite3
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from obspy import read, UTCDateTime

CATALOG_DB = Path('/Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite')
OUTPUT_ROOT = Path('/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_by_geode/qc_review')
PANEL_ROOT = OUTPUT_ROOT / 'review_panels'

CORR_THRESHOLD = 0.65
MAX_SHIFT_S = 0.08
BORDERLINE_HALF_WIDTH = 0.10
HIGH_REJECTION_FRACTION = 0.50
LOW_ACCEPTED_MEMBERS = 2
PANELS_PER_PRIORITY = {1: 5, 2: 10, 3: 8, 4: 5, 5: 5}
N_GOOD_AUDIT_PANELS = 6
WAVEFORM_TMIN_S = -0.05
WAVEFORM_TMAX_S = 0.75
PRIMARY_COMPONENT = 'Z'
RANDOM_SEED = 202605

COLORS = {
    'accepted': '#0072B2',
    'rejected': '#D55E00',
    'reference': '#111111',
    'threshold': '#666666',
    'failed': '#CC79A7',
    'good': '#009E73',
}

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PANEL_ROOT.mkdir(parents=True, exist_ok=True)
pd.set_option('display.max_columns', 100)
print('Catalog:', CATALOG_DB)
print('QC output:', OUTPUT_ROOT)

Catalog: /Volumes/tachyon/LBSSP_DATA/catalog/lbssp_shot_catalog.sqlite
QC output: /Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_by_geode/qc_review


## 1. Load notebook 94/95 products in immutable read-only mode

In [2]:
required_tables = [
    'nodal_stacks', 'nodal_stack_members', 'nodal_stack_files',
    'nodal_stack_processing_errors', 'nodal_stack_reference_candidates',
    'nodal_event_catalog_qc', 'geode_nodal_event_matches', 'trace_index',
]
uri = f'file:{CATALOG_DB}?mode=ro&immutable=1'
with sqlite3.connect(uri, uri=True) as conn:
    present = set(pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)['name'])
    missing = sorted(set(required_tables) - present)
    if missing:
        raise RuntimeError(f'Missing required tables; run notebooks 94 and 95 first: {missing}')
    stacks = pd.read_sql('SELECT * FROM nodal_stacks', conn)
    members = pd.read_sql('SELECT * FROM nodal_stack_members', conn)
    files = pd.read_sql('SELECT * FROM nodal_stack_files', conn)
    errors = pd.read_sql('SELECT * FROM nodal_stack_processing_errors', conn)
    refs = pd.read_sql('SELECT * FROM nodal_stack_reference_candidates', conn)
    events = pd.read_sql('SELECT * FROM nodal_event_catalog_qc', conn)
    matches = pd.read_sql('SELECT * FROM geode_nodal_event_matches', conn)
    trace_index = pd.read_sql("SELECT * FROM trace_index WHERE instrument_system='nodal'", conn)

for frame in (members, events, matches):
    for col in ('nodal_event_time_utc', 'geode_final_trigger_utc'):
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], utc=True, errors='coerce')

for col in ['accepted_for_stack', 'included_in_output_stack', 'is_reference_event']:
    if col in members.columns:
        members[col] = members[col].fillna(0).astype(bool)
if 'selected_reference' in refs.columns:
    refs['selected_reference'] = refs['selected_reference'].fillna(0).astype(bool)

print(f'{len(stacks):,} successful stacks')
print(f'{members.stack_id.nunique():,} processed stack groups')
print(f'{len(members):,} waveform-QC member decisions')
print(f'{(~members.accepted_for_stack).sum():,} rejected member events')
print(f'{len(errors):,} processing-error rows')

194 successful stacks
194 processed stack groups
961 waveform-QC member decisions
6 rejected member events
0 processing-error rows


## 2. Build ranked stack and event review queues

Priority 1 means the stack failed. Priority 2 has very few accepted members. Priority 3 rejected at least half of its candidate events. Priority 4 contains correlations close to the cutoff. Priority 5 is off the nodal line and deserves a geometry check.

In [3]:
geometry = (
    matches[['geode_event_id', 'source_geometry_class']]
    .dropna(subset=['geode_event_id'])
    .drop_duplicates('geode_event_id')
)
member_summary = (
    members.groupby('stack_id', as_index=False)
    .agg(
        geode_event_id=('geode_event_id', 'first'),
        geode_survey=('geode_survey', 'first'),
        line=('line', 'first'),
        file_no=('file_no', 'first'),
        source_x_truth_m=('source_x_truth_m', 'first'),
        stack_output_status=('stack_output_status', 'first'),
        n_candidate=('nodal_event_id', 'size'),
        n_accepted=('accepted_for_stack', 'sum'),
        n_rejected=('accepted_for_stack', lambda x: int((~x.astype(bool)).sum())),
        median_corr=('xcorr_corrcoef', 'median'),
        min_corr=('xcorr_corrcoef', 'min'),
        max_abs_shift_s=('xcorr_shift_s', lambda x: pd.to_numeric(x, errors='coerce').abs().max()),
    )
)
member_summary['rejection_fraction'] = member_summary.n_rejected / member_summary.n_candidate
borderline = members.assign(
    near_cutoff=(pd.to_numeric(members.xcorr_corrcoef, errors='coerce') - CORR_THRESHOLD).abs() <= BORDERLINE_HALF_WIDTH
).groupby('stack_id').near_cutoff.sum().rename('n_near_cutoff').reset_index()
max_rejected = (
    members.loc[~members.accepted_for_stack]
    .groupby('stack_id').xcorr_corrcoef.max().rename('max_rejected_corr').reset_index()
)
min_accepted = (
    members.loc[members.accepted_for_stack]
    .groupby('stack_id').xcorr_corrcoef.min().rename('min_accepted_corr').reset_index()
)
selected_refs = refs.loc[refs.selected_reference].copy()
selected_refs['reference_support_fraction'] = (
    selected_refs.n_supported_other_members / selected_refs.n_other_members_tested.replace(0, np.nan)
)
ref_summary = selected_refs[[
    'stack_id', 'reference_nodal_event_id', 'n_supported_other_members',
    'n_other_members_tested', 'reference_support_fraction', 'median_corr_to_other_members'
]]
stack_queue = (member_summary.merge(borderline, on='stack_id', how='left')
               .merge(max_rejected, on='stack_id', how='left')
               .merge(min_accepted, on='stack_id', how='left')
               .merge(ref_summary, on='stack_id', how='left')
               .merge(geometry, on='geode_event_id', how='left'))

def review_reasons(row):
    reasons = []
    if row.stack_output_status != 'stack_written': reasons.append('stack failed')
    if row.n_accepted <= LOW_ACCEPTED_MEMBERS: reasons.append(f'only {row.n_accepted} accepted')
    if row.rejection_fraction >= HIGH_REJECTION_FRACTION: reasons.append('>=50% rejected')
    if row.n_near_cutoff > 0: reasons.append(f'{int(row.n_near_cutoff)} near cutoff')
    if str(row.source_geometry_class).startswith('off_end_'): reasons.append(str(row.source_geometry_class))
    return '; '.join(reasons) if reasons else 'routine audit'

def review_priority(row):
    if row.stack_output_status != 'stack_written': return 1
    if row.n_accepted <= LOW_ACCEPTED_MEMBERS: return 2
    if row.rejection_fraction >= HIGH_REJECTION_FRACTION: return 3
    if row.n_near_cutoff > 0: return 4
    if str(row.source_geometry_class).startswith('off_end_'): return 5
    return 6

stack_queue['review_priority'] = stack_queue.apply(review_priority, axis=1)
stack_queue['review_reasons'] = stack_queue.apply(review_reasons, axis=1)
stack_queue = stack_queue.sort_values(
    ['review_priority', 'rejection_fraction', 'n_candidate'], ascending=[True, False, False]
).reset_index(drop=True)
stack_queue.insert(0, 'review_rank', np.arange(1, len(stack_queue) + 1))

event_queue = members.loc[
    (~members.accepted_for_stack)
    | (members.stack_output_status != 'stack_written')
    | ((pd.to_numeric(members.xcorr_corrcoef, errors='coerce') - CORR_THRESHOLD).abs() <= BORDERLINE_HALF_WIDTH)
].copy()
event_queue['distance_from_corr_cutoff'] = (
    pd.to_numeric(event_queue.xcorr_corrcoef, errors='coerce') - CORR_THRESHOLD
).abs()
event_queue = event_queue.merge(
    stack_queue[['stack_id', 'review_rank', 'review_priority', 'review_reasons', 'source_geometry_class']],
    on='stack_id', how='left'
).sort_values(['review_priority', 'distance_from_corr_cutoff', 'stack_id', 'nodal_event_time_utc'])

stack_queue.to_csv(OUTPUT_ROOT / '95b_stack_review_queue.csv', index=False)
event_queue.to_csv(OUTPUT_ROOT / '95b_event_review_queue.csv', index=False)
print('Stack priorities:')
print(stack_queue.review_priority.value_counts().sort_index())
display(stack_queue.head(25))

Stack priorities:
review_priority
2     11
3      1
4      1
5     17
6    164
Name: count, dtype: int64


,review_rank,stack_id,geode_event_id,geode_survey,line,file_no,source_x_truth_m,stack_output_status,n_candidate,n_accepted,n_rejected,median_corr,min_corr,max_abs_shift_s,rejection_fraction,n_near_cutoff,max_rejected_corr,min_accepted_corr,reference_nodal_event_id,n_supported_other_members,n_other_members_tested,reference_support_fraction,median_corr_to_other_members,source_geometry_class,review_priority,review_reasons
0,1,NODALSTACK_T1_T1_1m_refraction_F3010_x0092.5m,GEODE_T1_1M_REFRACTION_F3010,T1_1m_refraction,T1,3010,92.5,stack_written,2,2,0,0.986956,0.973912,0.114,0.0,0,NaN,0.973912,T1_N2_Refraction1m_T1_N2_E00035,1,1,1.000000,0.973912,inside_aperture,2,only 2 accepted
1,2,NODALSTACK_T1_T1_streamer_masw_F1004_x0091.5m,GEODE_T1_STREAMER_MASW_F1004,T1_streamer_masw,T1,1004,91.5,stack_written,2,2,0,0.981126,0.962251,0.000,0.0,0,NaN,0.962251,T1_N1_Streamer_T1_N1_E00154,1,1,1.000000,0.962251,inside_aperture,2,only 2 accepted
2,3,NODALSTACK_T1_T1_streamer_masw_F1008_x0097.5m,GEODE_T1_STREAMER_MASW_F1008,T1_streamer_masw,T1,1008,97.5,stack_written,2,2,0,0.977586,0.955171,0.008,0.0,0,NaN,0.955171,T1_N1_Streamer_T1_N1_E00169,1,1,1.000000,0.955171,inside_aperture,2,only 2 accepted
3,4,NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m,GEODE_T1_STREAMER_MASW_F1014,T1_streamer_masw,T1,1014,106.5,stack_written,2,2,0,0.969341,0.938683,0.020,0.0,0,NaN,0.938683,T1_N1_Streamer_T1_N1_E00194,1,1,1.000000,0.938683,inside_aperture,2,only 2 accepted
4,5,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,GEODE_T1_STREAMER_MASW_F1017,T1_streamer_masw,T1,1017,111.0,stack_written,2,2,0,0.985742,0.971485,0.000,0.0,0,NaN,0.971485,T1_N1_Streamer_T1_N1_E00205,1,1,1.000000,0.971485,inside_aperture,2,only 2 accepted
5,6,NODALSTACK_T1_T1_streamer_masw_F1022_x0118.5m,GEODE_T1_STREAMER_MASW_F1022,T1_streamer_masw,T1,1022,118.5,stack_written,2,2,0,0.996416,0.992833,0.190,0.0,0,NaN,0.992833,T1_N1_Streamer_T1_N1_E00228,1,1,1.000000,0.992833,inside_aperture,2,only 2 accepted
6,7,NODALSTACK_T1_T1_streamer_masw_F1024_x0121.5m,GEODE_T1_STREAMER_MASW_F1024,T1_streamer_masw,T1,1024,121.5,stack_written,2,2,0,0.990328,0.980656,0.028,0.0,0,NaN,0.980656,T1_N1_Streamer_T1_N1_E00240,1,1,1.000000,0.980656,inside_aperture,2,only 2 accepted
7,8,NODALSTACK_T1_T1_streamer_masw_F1028_x0127.5m,GEODE_T1_STREAMER_MASW_F1028,T1_streamer_masw,T1,1028,127.5,stack_written,2,2,0,0.967701,0.935401,0.032,0.0,0,NaN,0.935401,T1_N1_Streamer_T1_N1_E00258,1,1,1.000000,0.935401,inside_aperture,2,only 2 accepted
8,9,NODALSTACK_T1_T1_streamer_masw_F1032_x0133.5m,GEODE_T1_STREAMER_MASW_F1032,T1_streamer_masw,T1,1032,133.5,stack_written,2,2,0,0.990055,0.980109,0.112,0.0,0,NaN,0.980109,T1_N1_Streamer_T1_N1_E00274,1,1,1.000000,0.980109,inside_aperture,2,only 2 accepted
9,10,NODALSTACK_T1_T1_streamer_masw_F1035_x0138.0m,GEODE_T1_STREAMER_MASW_F1035,T1_streamer_masw,T1,1035,138.0,stack_written,2,2,0,0.992953,0.985906,0.002,0.0,0,NaN,0.985906,T1_N1_Streamer_T1_N1_E00288,1,1,1.000000,0.985906,inside_aperture,2,only 2 accepted


## 3. Campaign-wide QC figures

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10), constrained_layout=True)

# Accepted/rejected counts by survey.
counts = (members.assign(decision=np.where(members.accepted_for_stack, 'Accepted', 'Rejected'))
          .groupby(['geode_survey', 'decision']).size().unstack(fill_value=0))
counts = counts.reindex(columns=['Accepted', 'Rejected'], fill_value=0)
counts.plot.bar(stacked=True, ax=axes[0, 0], color=[COLORS['accepted'], COLORS['rejected']])
axes[0, 0].set(title='Waveform-QC decisions by Geode survey', xlabel='', ylabel='Candidate events')
axes[0, 0].tick_params(axis='x', rotation=25)

# Correlation distributions.
accepted_corr = pd.to_numeric(members.loc[members.accepted_for_stack, 'xcorr_corrcoef'], errors='coerce').dropna()
rejected_corr = pd.to_numeric(members.loc[~members.accepted_for_stack, 'xcorr_corrcoef'], errors='coerce').dropna()
bins = np.linspace(min(-0.1, rejected_corr.min()), 1.01, 40)
axes[0, 1].hist(accepted_corr, bins=bins, alpha=.70, color=COLORS['accepted'], label='Accepted')
axes[0, 1].hist(rejected_corr, bins=bins, alpha=.70, color=COLORS['rejected'], label='Rejected')
axes[0, 1].axvline(CORR_THRESHOLD, color=COLORS['threshold'], ls='--', lw=2, label=f'Cutoff {CORR_THRESHOLD:.2f}')
axes[0, 1].set(title='Event correlation with consensus reference', xlabel='Median cross-correlation coefficient', ylabel='Events')
axes[0, 1].legend()

# Candidate versus accepted count.
ok = stack_queue.stack_output_status == 'stack_written'
sc = axes[1, 0].scatter(stack_queue.loc[ok, 'n_candidate'], stack_queue.loc[ok, 'n_accepted'],
                        c=stack_queue.loc[ok, 'rejection_fraction'], cmap='viridis', vmin=0, vmax=1,
                        s=48, edgecolor='white', linewidth=.5)
axes[1, 0].scatter(stack_queue.loc[~ok, 'n_candidate'], stack_queue.loc[~ok, 'n_accepted'],
                   marker='X', s=100, color=COLORS['failed'], edgecolor='black', label='Stack failed')
lim = max(stack_queue.n_candidate.max(), stack_queue.n_accepted.max()) + 1
axes[1, 0].plot([0, lim], [0, lim], color='#aaaaaa', lw=1)
axes[1, 0].set(title='Candidates retained in each stack', xlabel='Candidate events', ylabel='Accepted events', xlim=(0, lim), ylim=(0, lim))
axes[1, 0].legend(loc='lower right')
fig.colorbar(sc, ax=axes[1, 0], label='Rejection fraction')

# Consensus-reference diagnostics.
refplot = stack_queue.dropna(subset=['reference_support_fraction', 'median_corr_to_other_members'])
ref_ok = refplot.stack_output_status == 'stack_written'
axes[1, 1].scatter(refplot.loc[ref_ok, 'reference_support_fraction'], refplot.loc[ref_ok, 'median_corr_to_other_members'],
                   color=COLORS['good'], alpha=.75, s=50, label='Stack written')
axes[1, 1].scatter(refplot.loc[~ref_ok, 'reference_support_fraction'], refplot.loc[~ref_ok, 'median_corr_to_other_members'],
                   marker='X', color=COLORS['failed'], edgecolor='black', s=100, label='Stack failed')
axes[1, 1].axhline(CORR_THRESHOLD, color=COLORS['threshold'], ls='--', lw=1.5)
axes[1, 1].set(title='Consensus-reference quality', xlabel='Fraction of other members supported', ylabel='Median correlation to other members', xlim=(-.03, 1.03))
axes[1, 1].legend()

fig.suptitle('Notebook 95 nodal stack QC overview', fontsize=16)
overview_path = OUTPUT_ROOT / '95b_qc_overview.png'
fig.savefig(overview_path, dpi=180, bbox_inches='tight')
plt.close(fig)
print(overview_path)

/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_by_geode/qc_review/95b_qc_overview.png


In [5]:
surveys = list(pd.unique(members.geode_survey.dropna()))
fig, axes = plt.subplots(len(surveys), 1, figsize=(15, 3.2 * len(surveys)), squeeze=False, constrained_layout=True)
for ax, survey in zip(axes[:, 0], surveys):
    d = members.loc[members.geode_survey == survey].sort_values('nodal_event_time_utc')
    accepted = d.accepted_for_stack
    ax.scatter(d.loc[accepted, 'nodal_event_time_utc'], d.loc[accepted, 'estimated_source_x_m'],
               s=18, color=COLORS['accepted'], alpha=.72, label='Accepted nodal estimate')
    ax.scatter(d.loc[~accepted, 'nodal_event_time_utc'], d.loc[~accepted, 'estimated_source_x_m'],
               s=28, marker='x', color=COLORS['rejected'], alpha=.9, label='Rejected nodal estimate')
    ax.scatter(d.nodal_event_time_utc, d.source_x_truth_m, s=14, facecolors='none', edgecolors='black',
               linewidth=.7, label='Known Geode source x')
    ax.set(title=str(survey), ylabel='Source x (m)')
    ax.grid(alpha=.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %H:%M'))
axes[-1, 0].set_xlabel('UTC event time')
axes[0, 0].legend(loc='upper left', fontsize=8, framealpha=.9)
fig.suptitle('Accepted and rejected nodal candidates versus known Geode source positions', fontsize=15)
position_path = OUTPUT_ROOT / '95b_accepted_rejected_position_time.png'
fig.savefig(position_path, dpi=180, bbox_inches='tight')
plt.close(fig)
print(position_path)

/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_by_geode/qc_review/95b_accepted_rejected_position_time.png


## 4. Generate per-stack waveform review panels

Each panel shows the position decisions, the correlation decisions, and aligned waveform overlays at two representative receivers. Rejected events that are visibly dissimilar support the automated rejection. A coherent orange waveform may indicate a threshold or reference problem worth manual review.

In [6]:
def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', str(value)).strip('_')

def attach_receiver_x(stream, event_id):
    geom = trace_index.loc[trace_index.event_id.astype(str) == str(event_id)].copy()
    geom = geom.loc[geom.channel.astype(str).str.endswith(PRIMARY_COMPONENT)]
    lookup = {}
    for _, row in geom.iterrows():
        if pd.isna(row.get('receiver_x_m')):
            continue
        station, channel = str(row.get('station', '')), str(row.get('channel', ''))
        lookup[(station, channel)] = float(row.receiver_x_m)
        if pd.notna(row.get('seed_id')):
            lookup[str(row.seed_id)] = float(row.receiver_x_m)
    out = stream.select(channel=f'*{PRIMARY_COMPONENT}').copy()
    for tr in out:
        x = lookup.get((tr.stats.station, tr.stats.channel), lookup.get(tr.id))
        if x is not None:
            tr.stats.receiver_x_m = x
    return out

def load_preprocessed_member(row):
    stream = attach_receiver_x(read(str(row.mseed_path)), row.nodal_event_id)
    for tr in stream:
        tr.data = tr.data.astype(np.float64)
        tr.detrend('linear')
        tr.taper(max_percentage=.05, type='hann')
        nyquist = .5 * tr.stats.sampling_rate
        high = min(150.0, .90 * nyquist)
        if high > 5.0:
            tr.filter('bandpass', freqmin=5.0, freqmax=high, corners=4, zerophase=True)
    return stream

def trace_at_receiver(stream, receiver_x):
    available = [tr for tr in stream if hasattr(tr.stats, 'receiver_x_m')]
    if not available:
        return None
    return min(available, key=lambda tr: abs(float(tr.stats.receiver_x_m) - receiver_x))

def aligned_normalized_waveform(tr, origin, shift_s, t_grid):
    rel = tr.times() + float(tr.stats.starttime - UTCDateTime(origin))
    y = np.interp(t_grid - float(shift_s), rel, tr.data.astype(float), left=np.nan, right=np.nan)
    scale = np.nanmax(np.abs(y))
    return y / scale if np.isfinite(scale) and scale > 0 else y

def representative_receivers(reference_stream, truth_x):
    xs = sorted({float(tr.stats.receiver_x_m) for tr in reference_stream if hasattr(tr.stats, 'receiver_x_m')})
    if not xs:
        return []
    nearest = min(xs, key=lambda x: abs(x - truth_x))
    middle = xs[len(xs) // 2]
    if middle == nearest and len(xs) > 1:
        middle = xs[0] if nearest != xs[0] else xs[-1]
    return [nearest, middle][:len(xs)]

def make_stack_panel(queue_row):
    stack_id = queue_row.stack_id
    d = members.loc[members.stack_id == stack_id].sort_values('nodal_event_time_utc').copy()
    loaded, failures = {}, []
    for _, row in d.iterrows():
        try:
            loaded[str(row.nodal_event_id)] = load_preprocessed_member(row)
        except Exception as exc:
            failures.append(f'{row.nodal_event_id}: {exc}')
    ref_rows = d.loc[d.is_reference_event]
    if ref_rows.empty or str(ref_rows.iloc[0].nodal_event_id) not in loaded:
        return None, failures + ['reference waveform unavailable']
    reference_stream = loaded[str(ref_rows.iloc[0].nodal_event_id)]
    receivers = representative_receivers(reference_stream, float(queue_row.source_x_truth_m))

    fig, axes = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
    accepted = d.accepted_for_stack
    axes[0, 0].scatter(d.loc[accepted, 'nodal_event_time_utc'], d.loc[accepted, 'estimated_source_x_m'],
                       color=COLORS['accepted'], s=45, label='Accepted')
    axes[0, 0].scatter(d.loc[~accepted, 'nodal_event_time_utc'], d.loc[~accepted, 'estimated_source_x_m'],
                       color=COLORS['rejected'], marker='x', s=65, label='Rejected')
    axes[0, 0].axhline(queue_row.source_x_truth_m, color='black', ls='--', label='Known Geode x')
    axes[0, 0].set(title='Nodal position estimates', ylabel='Source x (m)')
    axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter('%H:%M:%S'))
    axes[0, 0].legend(fontsize=8)

    order = np.arange(1, len(d) + 1)
    axes[0, 1].scatter(order[accepted.to_numpy()], d.loc[accepted, 'xcorr_corrcoef'], color=COLORS['accepted'], s=45, label='Accepted')
    axes[0, 1].scatter(order[(~accepted).to_numpy()], d.loc[~accepted, 'xcorr_corrcoef'], color=COLORS['rejected'], marker='x', s=65, label='Rejected')
    ref_mask = d.is_reference_event.to_numpy()
    axes[0, 1].scatter(order[ref_mask], d.loc[d.is_reference_event, 'xcorr_corrcoef'], facecolors='none', edgecolors='black', s=120, lw=1.8, label='Reference')
    axes[0, 1].axhline(CORR_THRESHOLD, color=COLORS['threshold'], ls='--', label=f'Cutoff {CORR_THRESHOLD:.2f}')
    axes[0, 1].set(title='Correlation decision by event order', xlabel='Candidate event order', ylabel='Correlation coefficient')
    axes[0, 1].legend(fontsize=8)

    t_grid = np.linspace(WAVEFORM_TMIN_S, WAVEFORM_TMAX_S, 900)
    for ax, receiver_x in zip(axes[1, :], receivers):
        for _, row in d.iterrows():
            stream = loaded.get(str(row.nodal_event_id))
            if stream is None:
                continue
            tr = trace_at_receiver(stream, receiver_x)
            if tr is None:
                continue
            shift = float(row.xcorr_shift_s) if pd.notna(row.xcorr_shift_s) else 0.0
            y = aligned_normalized_waveform(tr, row.nodal_event_time_utc, shift, t_grid)
            if row.is_reference_event:
                ax.plot(t_grid, y, color=COLORS['reference'], lw=2.0, alpha=.95, zorder=5)
            elif row.accepted_for_stack:
                ax.plot(t_grid, y, color=COLORS['accepted'], lw=1.0, alpha=.42)
            else:
                ax.plot(t_grid, y, color=COLORS['rejected'], lw=1.0, alpha=.60)
        ax.axvline(0, color='#999999', lw=.8)
        ax.set(title=f'Aligned normalized waveforms near receiver x={receiver_x:.1f} m', xlabel='Time from candidate origin (s)', ylabel='Normalized amplitude', xlim=(WAVEFORM_TMIN_S, WAVEFORM_TMAX_S))
        ax.grid(alpha=.18)
    for ax in axes[1, len(receivers):]:
        ax.axis('off')

    fig.suptitle(
        f"Review rank {int(queue_row.review_rank)} | {stack_id} | x={queue_row.source_x_truth_m:.1f} m | "
        f"{int(queue_row.n_accepted)}/{int(queue_row.n_candidate)} accepted\n{queue_row.review_reasons}",
        fontsize=14,
    )
    out = PANEL_ROOT / f"{int(queue_row.review_rank):03d}_{safe_name(stack_id)}.png"
    fig.savefig(out, dpi=165, bbox_inches='tight')
    plt.close(fig)
    return out, failures


In [7]:
flagged = pd.concat([
    stack_queue.loc[stack_queue.review_priority == priority].head(limit)
    for priority, limit in PANELS_PER_PRIORITY.items()
])
good_pool = stack_queue.loc[
    (stack_queue.review_priority == 6)
    & (stack_queue.rejection_fraction <= .20)
    & (stack_queue.median_corr >= .85)
]
good_audit = good_pool.sample(min(N_GOOD_AUDIT_PANELS, len(good_pool)), random_state=RANDOM_SEED)
panel_selection = pd.concat([flagged, good_audit]).drop_duplicates('stack_id')

manifest_rows = []
for _, row in panel_selection.iterrows():
    print(f"Panel {int(row.review_rank):03d}: {row.stack_id}")
    panel_path, failures = make_stack_panel(row)
    manifest_rows.append({
        'review_rank': row.review_rank,
        'review_priority': row.review_priority,
        'stack_id': row.stack_id,
        'review_reasons': row.review_reasons,
        'panel_path': str(panel_path) if panel_path else None,
        'waveform_load_warnings': ' | '.join(failures),
    })
manifest = pd.DataFrame(manifest_rows).sort_values(['review_priority', 'review_rank'])
manifest.to_csv(OUTPUT_ROOT / '95b_stack_panel_manifest.csv', index=False)
stack_queue = stack_queue.merge(manifest[['stack_id', 'panel_path']], on='stack_id', how='left')
stack_queue.to_csv(OUTPUT_ROOT / '95b_stack_review_queue.csv', index=False)
print(f'Wrote {manifest.panel_path.notna().sum()} stack panels')
display(manifest.head(40))

Panel 001: NODALSTACK_T1_T1_1m_refraction_F3010_x0092.5m
Panel 002: NODALSTACK_T1_T1_streamer_masw_F1004_x0091.5m
Panel 003: NODALSTACK_T1_T1_streamer_masw_F1008_x0097.5m
Panel 004: NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m
Panel 005: NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m
Panel 006: NODALSTACK_T1_T1_streamer_masw_F1022_x0118.5m
Panel 007: NODALSTACK_T1_T1_streamer_masw_F1024_x0121.5m
Panel 008: NODALSTACK_T1_T1_streamer_masw_F1028_x0127.5m
Panel 009: NODALSTACK_T1_T1_streamer_masw_F1032_x0133.5m
Panel 010: NODALSTACK_T1_T1_streamer_masw_F1035_x0138.0m
Panel 012: NODALSTACK_T3_T3_1m_refraction_F4022_x0041.5m
Panel 013: NODALSTACK_T1_T1_streamer_masw_F1069_x0189.0m
Panel 014: NODALSTACK_T1_T1_2m_refraction_F3053_x0067.0m
Panel 015: NODALSTACK_T1_T1_2m_refraction_F3083_x0187.0m
Panel 016: NODALSTACK_T3_T3_1m_refraction_F4038_x0073.5m
Panel 017: NODALSTACK_T1_T1_2m_refraction_F3048_x0047.0m
Panel 018: NODALSTACK_T1_T1_2m_refraction_F3052_x0063.0m
Panel 165: NODALSTACK_T1_T1_str

,review_rank,review_priority,stack_id,review_reasons,panel_path,waveform_load_warnings
0,1,2,NODALSTACK_T1_T1_1m_refraction_F3010_x0092.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
1,2,2,NODALSTACK_T1_T1_streamer_masw_F1004_x0091.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
2,3,2,NODALSTACK_T1_T1_streamer_masw_F1008_x0097.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
3,4,2,NODALSTACK_T1_T1_streamer_masw_F1014_x0106.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
4,5,2,NODALSTACK_T1_T1_streamer_masw_F1017_x0111.0m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
5,6,2,NODALSTACK_T1_T1_streamer_masw_F1022_x0118.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
6,7,2,NODALSTACK_T1_T1_streamer_masw_F1024_x0121.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
7,8,2,NODALSTACK_T1_T1_streamer_masw_F1028_x0127.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
8,9,2,NODALSTACK_T1_T1_streamer_masw_F1032_x0133.5m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,
9,10,2,NODALSTACK_T1_T1_streamer_masw_F1035_x0138.0m,only 2 accepted,/Volumes/tachyon/LBSSP_DATA/95_nodal_stacked_b...,


## 5. Review summary and suggested order

In [8]:
summary = pd.DataFrame([
    ('Processed stack groups', members.stack_id.nunique()),
    ('Successful output stacks', len(stacks)),
    ('Failed stack groups', (stack_queue.stack_output_status != 'stack_written').sum()),
    ('Waveform-accepted candidate events', members.accepted_for_stack.sum()),
    ('Included in successful output stacks', members.included_in_output_stack.sum()),
    ('Rejected candidate events', (~members.accepted_for_stack).sum()),
    ('Stacks rejecting at least half', (stack_queue.rejection_fraction >= HIGH_REJECTION_FRACTION).sum()),
    ('Stacks with near-cutoff events', (stack_queue.n_near_cutoff > 0).sum()),
    ('Off-end stack groups', stack_queue.source_geometry_class.astype(str).str.startswith('off_end_').sum()),
    ('Generated review panels', manifest.panel_path.notna().sum()),
], columns=['metric', 'value'])
summary.to_csv(OUTPUT_ROOT / '95b_qc_summary.csv', index=False)
display(summary)
print('Review in this order:')
print('1. 95b_qc_overview.png')
print('2. 95b_accepted_rejected_position_time.png')
print('3. 95b_stack_review_queue.csv, starting at review_rank 1')
print('4. review_panels/*.png; coherent rejected waveforms deserve manual attention')


,metric,value
0,Processed stack groups,194
1,Successful output stacks,194
2,Failed stack groups,0
3,Waveform-accepted candidate events,955
4,Included in successful output stacks,955
5,Rejected candidate events,6
6,Stacks rejecting at least half,1
7,Stacks with near-cutoff events,1
8,Off-end stack groups,17
9,Generated review panels,23


Review in this order:
1. 95b_qc_overview.png
2. 95b_accepted_rejected_position_time.png
3. 95b_stack_review_queue.csv, starting at review_rank 1
4. review_panels/*.png; coherent rejected waveforms deserve manual attention
